In [5]:
import cv2
import os
import time
import tkinter as tk
from tkinter import filedialog, simpledialog
from datetime import datetime, timedelta

def capture_and_record_cctv():
    # --- 1. GUI 입력 창 ---
    root = tk.Tk()
    root.withdraw()

    m3u8_url = simpledialog.askstring("CCTV 연결", "CCTV m3u8 주소를 입력하세요:", 
                                    initialvalue="https://cctvsec.ktict.co.kr:8091/2431/iV39o9nizyCQ0Df...")
    if not m3u8_url: return

    selected_path = filedialog.askdirectory(title="저장할 폴더 선택")
    if not selected_path: return

    # --- [추가 기능] 예약 시간 설정 ---
    # 예: 오늘 밤 11시에 눈이 온다면 "23:00" 입력
    start_time_str = simpledialog.askstring("예약 녹화", "몇 시에 녹화를 시작할까요? (24시간제, 예: 23:00)\n지금 바로 하려면 취소 또는 엔터", initialvalue="")
    
    # 녹화할 시간(분) 설정
    duration_min = simpledialog.askinteger("녹화 시간", "몇 분 동안 녹화할까요?", initialvalue=10, minvalue=1)

    # --- 대기 모드 진입 ---
    if start_time_str:
        now = datetime.now()
        target_time = datetime.strptime(start_time_str, "%H:%M").replace(year=now.year, month=now.month, day=now.day)
        
        # 만약 입력한 시간이 현재보다 이전이면 내일로 설정 (예: 지금 23시인데 02시 입력 시)
        if target_time < now:
            target_time += timedelta(days=1)

        wait_seconds = (target_time - now).total_seconds()
        print(f"\n⏰ 예약됨: {target_time.strftime('%Y-%m-%d %H:%M:%S')}에 녹화를 시작합니다.")
        print(f"남은 시간: {int(wait_seconds//3600)}시간 {int((wait_seconds%3600)//60)}분")
        print("프로그램을 끄지 말고 기다려주세요...")
        
        time.sleep(wait_seconds) # 해당 시간까지 코드 잠재우기

    # --- 2. 폴더 생성 (녹화 시작 시점의 시간으로) ---
    current_time_folder = time.strftime("%Y-%m-%d_%H-%M-%S") # 실제 녹화 시작 시간
    output_base_path = os.path.join(selected_path, current_time_folder)
    
    img_folder = os.path.join(output_base_path, "frames")
    video_folder = os.path.join(output_base_path, "videos")
    os.makedirs(img_folder, exist_ok=True)
    os.makedirs(video_folder, exist_ok=True)

    # --- 3. 녹화 시작 ---
    cap = cv2.VideoCapture(m3u8_url)
    if not cap.isOpened():
        print("❌ CCTV 연결 실패 (URL이 만료되었거나 인터넷 문제)")
        return

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or fps > 100: fps = 30

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_filename = f"record_{current_time_folder}.mp4"
    save_path = os.path.join(video_folder, video_filename)
    video_writer = cv2.VideoWriter(save_path, fourcc, fps, (width, height))

    interval = int(fps * 5) # 5초마다 캡쳐
    frame_count = 0
    saved_img_count = 0
    
    # 종료 시간 계산
    start_record_time = time.time()
    end_record_time = start_record_time + (duration_min * 60)

    print(f"\n🎥 녹화 시작! (저장 경로: {output_base_path})")
    print(f"⏱️ {duration_min}분 뒤에 자동으로 종료됩니다.")

    try:
        while True:
            # 설정한 시간이 지나면 자동 종료
            if time.time() > end_record_time:
                print("\n✅ 설정한 녹화 시간이 끝났습니다.")
                break

            ret, frame = cap.read()
            if not ret: 
                print("신호 끊김")
                break

            video_writer.write(frame)

            # if frame_count % interval == 0:
            #     img_name = f"snap_{saved_img_count:04d}.jpg"
            #     cv2.imwrite(os.path.join(img_folder, img_name), frame)
            #     saved_img_count += 1

            cv2.imshow("CCTV Recording...", frame)

            frame_count += 1
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    except KeyboardInterrupt:
        print("\n강제 중지됨")
    finally:
        cap.release()
        video_writer.release()
        cv2.destroyAllWindows()
        root.destroy()
        print(f"파일 저장 완료: {save_path}")

if __name__ == "__main__":
    capture_and_record_cctv()


🎥 녹화 시작! (저장 경로: N:/개인/대원&수빈/최종 프로젝트/임시\2026-03-31_10-44-07)
⏱️ 10분 뒤에 자동으로 종료됩니다.

✅ 설정한 녹화 시간이 끝났습니다.
파일 저장 완료: N:/개인/대원&수빈/최종 프로젝트/임시\2026-03-31_10-44-07\videos\record_2026-03-31_10-44-07.mp4
